In [0]:
%pip install ruamel.yaml openapi-spec-validator strands-agents python-dotenv openai -q

In [0]:
%restart_python

In [0]:
import sys, os, importlib

# Add the Standard-Agent directory to sys.path
project_dir = "/Workspace/Users/edward.m.ruiz@brighthousefinancial.com/Standard-Agent"
if project_dir not in sys.path:
    sys.path.insert(0, project_dir)

# Get auth from Databricks SDK (works on serverless, clusters, and apps)
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()
auth = w.config.authenticate()

# SDK >=0.67 returns a dict; older versions return a callable
if callable(auth):
    headers = {}
    auth(headers)
else:
    headers = auth

token = headers.get("Authorization", "").removeprefix("Bearer ")
host = w.config.host

# Set env vars that the portable agent reads
os.environ["OPENAI_API_KEY"] = token
os.environ["OPENAI_BASE_URL"] = f"{host}/serving-endpoints"
os.environ["MODEL_ID"] = "databricks-claude-sonnet-4"

print(f"Host: {host}")
print(f"Token: {'set' if token else 'NOT SET'} ({len(token)} chars)")
print(f"Model: {os.environ['MODEL_ID']}")
print()

# Force-reload ALL agent modules to pick up changes
import agent.tools
importlib.reload(agent.tools)
import agent.system_prompt
importlib.reload(agent.system_prompt)
import agent.agent
importlib.reload(agent.agent)

from agent.agent import create_agent

iri_agent = create_agent()
print("\n\u2705 Agent ready for testing")

In [0]:
# Test: GitHub URL-based spec review
# The agent should detect the URL, use fetch_yaml_from_url to download it,
# then proceed with the standard review workflow.

databricks = "/Workspace/Users/edward.m.ruiz@brighthousefinancial.com/Standard-Agent/draft-api-specs"

question = f"Review this spec for style guide compliance: {databricks}\nThis is a revision — no previous findings to compare. Yes, check cross-spec consistency."

result = iri_agent(question)
print(str(result))

In [0]:
# After the review above, ask the agent to generate a corrected YAML.
# The agent remembers the spec + findings from the previous call (conversation history).
# Just send a follow-up message — no need to re-pass the URL or YAML.

fix_prompt = """
Based on your review findings above, generate a corrected version of the 
FundTransfer YAML that addresses ALL Critical and Moderate issues:

- Fix all STRUCT findings (missing required arrays, oneOf issues, etc.)
- Fix all STYLE findings (policyNumber pattern, 202 description, etc.)
- Keep the spec functionally equivalent — don't remove endpoints or fields
- Output the complete corrected YAML
"""

fixed_result = iri_agent(fix_prompt)
print(str(fixed_result))

In [0]:
# Optional: Save the corrected YAML to a file and re-validate it.
# Extract the YAML block from the agent's response, then ask for a clean re-review.

# Step 1 — Ask the agent to output ONLY the raw YAML (no commentary)
raw_yaml_result = iri_agent(
    "Output ONLY the corrected YAML — no markdown fences, no explanation, "
    "just the raw YAML content starting with 'openapi:'"
)

# Step 2 — Save to file
output_path = "/Workspace/Users/edward.m.ruiz@brighthousefinancial.com/Standard-Agent/output"
os.makedirs(output_path, exist_ok=True)

yaml_text = str(raw_yaml_result)
with open(f"{output_path}/FundTransfer_v1.1.0_corrected.yml", "w") as f:
    f.write(yaml_text)
print(f"✅ Saved to {output_path}/FundTransfer_v1.1.0_corrected.yml")
print(f"   Size: {len(yaml_text):,} chars")

# Step 3 — Re-validate the corrected spec (new agent = clean slate)
print("\n🔄 Re-validating corrected spec...")
fresh_agent = create_agent()
review2 = fresh_agent(
    f"Review this YAML for style guide compliance. "
    f"Yes, check cross-spec consistency.\n\n{yaml_text}"
)
print(str(review2))